# Understanding Clouds from Satellite Images - FPN with ResNet

This notebook implements **Feature Pyramid Network (FPN) with ResNet50 backbone** for cloud segmentation.

## Why FPN + ResNet?
- **FPN:** Multi-scale feature extraction (detects clouds at different sizes)
- **ResNet50:** Pre-trained on ImageNet (transfer learning for faster convergence)
- **Speed:** Faster than U-Net while maintaining accuracy
- **Balance:** Good trade-off between performance and computational cost

## Problem:
Segment 4 cloud types in satellite images:
1. Fish
2. Flower
3. Gravel
4. Sugar

# Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Configuration and Paths

In [ ]:
# Paths
work_dir = "/kaggle/working/"
data_dir = "../input/understanding_cloud_organization"
train_csv_path = os.path.join(data_dir, 'train.csv')
train_image_path = os.path.join(data_dir, 'train_images')
test_image_path = os.path.join(data_dir, 'test_images')

# Hyperparameters
IMG_HEIGHT = 320  # Reduced from 1400 for faster training
IMG_WIDTH = 480   # Reduced from 2100
BATCH_SIZE = 8
EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 4   # Fish, Flower, Gravel, Sugar

# Cloud types
CLOUD_TYPES = ['Fish', 'Flower', 'Gravel', 'Sugar']

# Load and Prepare Data

In [ ]:
# Load training CSV
train_df = pd.read_csv(train_csv_path).fillna(-1)
print(f"Original shape: {train_df.shape}")
train_df.head()

In [ ]:
# Extract Image_Id and Label from Image_Label column
train_df['Image_Id'] = train_df['Image_Label'].apply(lambda x: x.split('_')[0])
train_df['Label'] = train_df['Image_Label'].apply(lambda x: x.split('_')[1])

# Create tuple of (Label, EncodedPixels)
train_df['Label_EncodedPixels'] = train_df.apply(
    lambda row: (row['Label'], row['EncodedPixels']), axis=1
)

train_df.head()

In [ ]:
# Group by Image_Id to get all masks for each image
grouped_df = train_df.groupby('Image_Id')['Label_EncodedPixels'].apply(list)
train_df = grouped_df.to_frame().reset_index()

print(f"Grouped shape: {train_df.shape}")
train_df.head()

In [ ]:
# Create binary columns for each cloud type
for label in CLOUD_TYPES:
    train_df[label] = 0

for index, row in train_df.iterrows():
    for item in row['Label_EncodedPixels']:
        label, value = item
        if value != -1:
            train_df.loc[index, label] = 1

# Create classes column (list of present cloud types)
train_df['classes'] = train_df.apply(
    lambda row: [col for col in CLOUD_TYPES if row[col] == 1], axis=1
)

print(f"\nCloud type distribution:")
for cloud_type in CLOUD_TYPES:
    count = train_df[cloud_type].sum()
    print(f"{cloud_type}: {count} images ({count/len(train_df)*100:.1f}%)")

train_df.head()

# Helper Functions for Data Processing

In [ ]:
def rle_to_mask(rle_string, height=1400, width=2100):
    """
    Converts Run-Length Encoding (RLE) to binary mask.
    
    RLE format: "start1 length1 start2 length2 ..."
    Example: "1 3 10 5" means pixels 1,2,3 and 10,11,12,13,14 are cloud
    """
    if rle_string == -1:
        return np.zeros((height, width), dtype=np.uint8)
    
    # Parse RLE string
    s = rle_string.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1  # Convert to 0-indexed
    ends = starts + lengths
    
    # Create mask
    mask = np.zeros(height * width, dtype=np.uint8)
    for start, end in zip(starts, ends):
        mask[start:end] = 1
    
    return mask.reshape((width, height)).T  # Fortran order

# Test the function
test_rle = "1 5 20 10"
test_mask = rle_to_mask(test_rle, height=10, width=10)
print(f"Test mask shape: {test_mask.shape}")
print(f"Test mask sum: {test_mask.sum()} (should be 15)")

In [ ]:
def get_masks_by_img_id(df, image_id, original_height=1400, original_width=2100):
    """
    Get all 4 cloud masks for a given image.
    Returns: (height, width, 4) array
    """
    row = df[df['Image_Id'] == image_id].iloc[0]
    masks = np.zeros((original_height, original_width, NUM_CLASSES), dtype=np.uint8)
    
    for idx, (label, encoded_pixels) in enumerate(row['Label_EncodedPixels']):
        mask = rle_to_mask(encoded_pixels, original_height, original_width)
        
        # Assign to correct channel based on cloud type
        cloud_idx = CLOUD_TYPES.index(label)
        masks[:, :, cloud_idx] = mask
    
    return masks

In [ ]:
def load_and_preprocess_image(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH)):
    """
    Load image and resize to target size.
    Normalizes pixel values to [0, 1].
    """
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (target_size[1], target_size[0]))  # (width, height)
    img = img.astype(np.float32) / 255.0
    return img

# Visualize Sample Images with Masks

In [ ]:
def visualize_sample(df, image_path, idx=0):
    """
    Visualize image with all 4 cloud masks.
    """
    image_id = df.iloc[idx]['Image_Id']
    img_file = os.path.join(image_path, image_id)
    
    # Load image
    img = cv2.imread(img_file)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Get masks
    masks = get_masks_by_img_id(df, image_id)
    
    # Plot
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    # Original image
    axes[0].imshow(img)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Individual masks
    colors = ['Reds', 'Blues', 'Greens', 'Purples']
    for i, (cloud_type, cmap) in enumerate(zip(CLOUD_TYPES, colors)):
        axes[i+1].imshow(img)
        axes[i+1].imshow(masks[:, :, i], alpha=0.5, cmap=cmap)
        axes[i+1].set_title(f'{cloud_type} Mask')
        axes[i+1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize first sample
visualize_sample(train_df, train_image_path, idx=18)

# Data Augmentation

In [ ]:
# Albumentations augmentation pipeline
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
    ], p=0.5),
    A.GaussNoise(p=0.3),
])

# No augmentation for validation
val_transform = None

print("Augmentation pipeline created!")

# Data Generator

In [ ]:
class CloudDataGenerator(keras.utils.Sequence):
    """
    Custom data generator for cloud segmentation.
    Loads images and masks on-the-fly to save memory.
    """
    def __init__(self, df, image_dir, batch_size=8, target_size=(320, 480), 
                 augmentation=None, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.target_size = target_size
        self.augmentation = augmentation
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))
    
    def __getitem__(self, index):
        # Get batch indexes
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        
        # Generate data
        X, y = self.__data_generation(batch_indexes)
        return X, y
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __data_generation(self, batch_indexes):
        # Initialize arrays
        X = np.zeros((len(batch_indexes), *self.target_size, 3), dtype=np.float32)
        y = np.zeros((len(batch_indexes), *self.target_size, NUM_CLASSES), dtype=np.float32)
        
        for i, idx in enumerate(batch_indexes):
            # Load image
            image_id = self.df.iloc[idx]['Image_Id']
            img_path = os.path.join(self.image_dir, image_id)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Load masks
            masks = get_masks_by_img_id(self.df, image_id)
            
            # Apply augmentation
            if self.augmentation:
                augmented = self.augmentation(image=img, mask=masks)
                img = augmented['image']
                masks = augmented['mask']
            
            # Resize
            img = cv2.resize(img, (self.target_size[1], self.target_size[0]))
            masks = cv2.resize(masks, (self.target_size[1], self.target_size[0]))
            
            # Normalize
            img = img.astype(np.float32) / 255.0
            
            # Handle mask shape after resize
            if len(masks.shape) == 2:
                masks = masks[..., np.newaxis]
            
            X[i] = img
            y[i] = masks
        
        return X, y

# Train-Validation Split

In [ ]:
# Split data: 80% train, 20% validation
train_df_split, val_df_split = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42,
    stratify=train_df['classes'].apply(lambda x: len(x))  # Stratify by number of cloud types
)

print(f"Training samples: {len(train_df_split)}")
print(f"Validation samples: {len(val_df_split)}")

In [ ]:
# Create data generators
train_generator = CloudDataGenerator(
    train_df_split,
    train_image_path,
    batch_size=BATCH_SIZE,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    augmentation=train_transform,
    shuffle=True
)

val_generator = CloudDataGenerator(
    val_df_split,
    train_image_path,
    batch_size=BATCH_SIZE,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    augmentation=None,
    shuffle=False
)

print(f"Training batches: {len(train_generator)}")
print(f"Validation batches: {len(val_generator)}")

# Build FPN with ResNet50 Backbone

## Architecture Overview:
```
Input Image (320x480x3)
    ↓
ResNet50 Encoder (pretrained on ImageNet)
    ├─ C1: 160×240×64   (stride 2)
    ├─ C2: 80×120×256   (stride 4)
    ├─ C3: 40×60×512    (stride 8)
    ├─ C4: 20×30×1024   (stride 16)
    └─ C5: 10×15×2048   (stride 32)
    ↓
FPN (Feature Pyramid Network)
    ├─ P5: 10×15×256    (top-down)
    ├─ P4: 20×30×256    (P5 + C4)
    ├─ P3: 40×60×256    (P4 + C3)
    └─ P2: 80×120×256   (P3 + C2)
    ↓
Segmentation Head
    └─ Upsample to 320×480×4 (4 cloud types)
```

## Why This Works:
- **Bottom-up pathway (ResNet):** Extracts features at multiple scales
- **Top-down pathway (FPN):** Combines low-level details with high-level semantics
- **Lateral connections:** Merge features from different levels

In [ ]:
def build_fpn_resnet50(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):
    """
    Build FPN with ResNet50 backbone.
    
    Args:
        input_shape: Input image size
        num_classes: Number of segmentation classes
    
    Returns:
        Keras model
    """
    
    # Input
    inputs = layers.Input(shape=input_shape)
    
    # ----- ENCODER (ResNet50 Backbone) -----
    # Load pretrained ResNet50 (without top classification layer)
    resnet = ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    
    # Extract feature maps from different stages
    # C2, C3, C4, C5 correspond to different ResNet blocks
    c1 = resnet.get_layer('conv1_relu').output          # 160×240×64
    c2 = resnet.get_layer('conv2_block3_out').output    # 80×120×256
    c3 = resnet.get_layer('conv3_block4_out').output    # 40×60×512
    c4 = resnet.get_layer('conv4_block6_out').output    # 20×30×1024
    c5 = resnet.get_layer('conv5_block3_out').output    # 10×15×2048
    
    # ----- FPN (Feature Pyramid Network) -----
    # Top-down pathway with lateral connections
    
    # P5: Start from highest level (smallest spatial dimensions)
    p5 = layers.Conv2D(256, (1, 1), padding='same', name='fpn_c5p5')(c5)
    
    # P4: Upsample P5 and add to C4
    p5_upsampled = layers.UpSampling2D(size=(2, 2), name='fpn_p5upsampled')(p5)
    c4_reduced = layers.Conv2D(256, (1, 1), padding='same', name='fpn_c4p4')(c4)
    p4 = layers.Add(name='fpn_p4add')([p5_upsampled, c4_reduced])
    p4 = layers.Conv2D(256, (3, 3), padding='same', name='fpn_p4')(p4)
    
    # P3: Upsample P4 and add to C3
    p4_upsampled = layers.UpSampling2D(size=(2, 2), name='fpn_p4upsampled')(p4)
    c3_reduced = layers.Conv2D(256, (1, 1), padding='same', name='fpn_c3p3')(c3)
    p3 = layers.Add(name='fpn_p3add')([p4_upsampled, c3_reduced])
    p3 = layers.Conv2D(256, (3, 3), padding='same', name='fpn_p3')(p3)
    
    # P2: Upsample P3 and add to C2
    p3_upsampled = layers.UpSampling2D(size=(2, 2), name='fpn_p3upsampled')(p3)
    c2_reduced = layers.Conv2D(256, (1, 1), padding='same', name='fpn_c2p2')(c2)
    p2 = layers.Add(name='fpn_p2add')([p3_upsampled, c2_reduced])
    p2 = layers.Conv2D(256, (3, 3), padding='same', name='fpn_p2')(p2)
    
    # ----- DECODER (Segmentation Head) -----
    # Combine all pyramid levels
    # Upsample all to same size as P2
    p3_up = layers.UpSampling2D(size=(2, 2), interpolation='bilinear')(p3)
    p4_up = layers.UpSampling2D(size=(4, 4), interpolation='bilinear')(p4)
    p5_up = layers.UpSampling2D(size=(8, 8), interpolation='bilinear')(p5)
    
    # Concatenate all levels
    merged = layers.Concatenate()([p2, p3_up, p4_up, p5_up])
    
    # Refine features
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(merged)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Upsample to original size (320×480)
    x = layers.UpSampling2D(size=(4, 4), interpolation='bilinear')(x)
    
    # Final segmentation layer
    outputs = layers.Conv2D(num_classes, (1, 1), activation='sigmoid', name='output')(x)
    
    # Create model
    model = models.Model(inputs=inputs, outputs=outputs, name='FPN_ResNet50')
    
    return model

In [ ]:
# Build model
model = build_fpn_resnet50(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    num_classes=NUM_CLASSES
)

# Print model summary
model.summary()

# Count parameters
total_params = model.count_params()
print(f"\nTotal parameters: {total_params:,}")

# Loss Functions and Metrics

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Dice Coefficient: Measures overlap between prediction and ground truth.
    Formula: 2 * |X ∩ Y| / (|X| + |Y|)
    Range: [0, 1] where 1 is perfect overlap
    """
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    """
    Dice Loss: 1 - Dice Coefficient
    Used for segmentation tasks with imbalanced classes.
    """
    return 1 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    """
    Combined Binary Cross-Entropy + Dice Loss.
    - BCE: Pixel-wise classification loss
    - Dice: Region-based overlap loss
    """
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

def iou_score(y_true, y_pred, smooth=1e-6):
    """
    IoU (Intersection over Union): Jaccard Index
    Formula: |X ∩ Y| / |X ∪ Y|
    """
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

# Compile Model

In [ ]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=[dice_coefficient, iou_score, 'binary_accuracy']
)

print("Model compiled successfully!")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Loss: Binary Cross-Entropy + Dice Loss")
print(f"Metrics: Dice Coefficient, IoU, Binary Accuracy")

# Callbacks

In [ ]:
# Setup callbacks
callbacks = [
    # Save best model based on validation Dice coefficient
    ModelCheckpoint(
        'best_fpn_resnet50.h5',
        monitor='val_dice_coefficient',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Stop training if no improvement for 10 epochs
    EarlyStopping(
        monitor='val_dice_coefficient',
        mode='max',
        patience=10,
        verbose=1,
        restore_best_weights=True
    ),
    
    # Reduce learning rate if plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print("1. ModelCheckpoint - Save best model")
print("2. EarlyStopping - Stop if no improvement (patience=10)")
print("3. ReduceLROnPlateau - Reduce LR on plateau (patience=5)")

# Train Model

In [ ]:
# Train model
print("Starting training...\n")

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed!")

# Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss')
axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Loss (BCE + Dice)', fontsize=14)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Dice Coefficient
axes[0, 1].plot(history.history['dice_coefficient'], label='Train Dice')
axes[0, 1].plot(history.history['val_dice_coefficient'], label='Val Dice')
axes[0, 1].set_title('Dice Coefficient', fontsize=14)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Dice Score')
axes[0, 1].legend()
axes[0, 1].grid(True)

# IoU Score
axes[1, 0].plot(history.history['iou_score'], label='Train IoU')
axes[1, 0].plot(history.history['val_iou_score'], label='Val IoU')
axes[1, 0].set_title('IoU Score', fontsize=14)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('IoU')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Binary Accuracy
axes[1, 1].plot(history.history['binary_accuracy'], label='Train Acc')
axes[1, 1].plot(history.history['val_binary_accuracy'], label='Val Acc')
axes[1, 1].set_title('Binary Accuracy', fontsize=14)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.show()

# Evaluate Model

In [ ]:
# Evaluate on validation set
print("Evaluating model on validation set...\n")

val_loss, val_dice, val_iou, val_acc = model.evaluate(val_generator, verbose=1)

print(f"\n{'='*50}")
print("VALIDATION RESULTS")
print(f"{'='*50}")
print(f"Loss (BCE + Dice): {val_loss:.4f}")
print(f"Dice Coefficient:  {val_dice:.4f}")
print(f"IoU Score:         {val_iou:.4f}")
print(f"Binary Accuracy:   {val_acc:.4f}")
print(f"{'='*50}")

# Visualize Predictions

In [ ]:
def visualize_predictions(generator, model, num_samples=3):
    """
    Visualize model predictions on validation samples.
    """
    # Get a batch
    X_batch, y_batch = generator[0]
    
    # Predict
    y_pred = model.predict(X_batch[:num_samples])
    
    # Visualize
    for i in range(num_samples):
        fig, axes = plt.subplots(2, 5, figsize=(25, 10))
        
        # Original image
        axes[0, 0].imshow(X_batch[i])
        axes[0, 0].set_title('Input Image', fontsize=14)
        axes[0, 0].axis('off')
        
        axes[1, 0].imshow(X_batch[i])
        axes[1, 0].set_title('Input Image', fontsize=14)
        axes[1, 0].axis('off')
        
        # Individual cloud types
        colors = ['Reds', 'Blues', 'Greens', 'Purples']
        
        for j, (cloud_type, cmap) in enumerate(zip(CLOUD_TYPES, colors)):
            # Ground truth
            axes[0, j+1].imshow(X_batch[i])
            axes[0, j+1].imshow(y_batch[i, :, :, j], alpha=0.5, cmap=cmap)
            axes[0, j+1].set_title(f'{cloud_type} (Ground Truth)', fontsize=14)
            axes[0, j+1].axis('off')
            
            # Prediction
            pred_mask = (y_pred[i, :, :, j] > 0.5).astype(np.float32)
            axes[1, j+1].imshow(X_batch[i])
            axes[1, j+1].imshow(pred_mask, alpha=0.5, cmap=cmap)
            axes[1, j+1].set_title(f'{cloud_type} (Prediction)', fontsize=14)
            axes[1, j+1].axis('off')
        
        plt.tight_layout()
        plt.savefig(f'prediction_sample_{i}.png', dpi=100, bbox_inches='tight')
        plt.show()

# Visualize predictions
visualize_predictions(val_generator, model, num_samples=3)

# Mask to RLE Encoding (for Submission)

In [ ]:
def mask_to_rle(mask):
    """
    Convert binary mask to Run-Length Encoding (RLE).
    
    Args:
        mask: 2D binary mask (height, width)
    
    Returns:
        RLE string or empty string if no mask
    """
    # Flatten mask in Fortran order
    pixels = mask.T.flatten()
    
    # No mask present
    if pixels.sum() == 0:
        return ''
    
    # Find run starts and lengths
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    
    return ' '.join(str(x) for x in runs)

# Make Predictions on Test Set

In [ ]:
# Load test images
test_images = sorted(os.listdir(test_image_path))
print(f"Number of test images: {len(test_images)}")

# Create submission dataframe
submission_rows = []

print("Generating predictions for test set...")

for img_name in test_images:
    # Load and preprocess image
    img_path = os.path.join(test_image_path, img_name)
    img = load_and_preprocess_image(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_batch = np.expand_dims(img, axis=0)
    
    # Predict
    pred_mask = model.predict(img_batch, verbose=0)[0]
    
    # Resize back to original size (1400x2100)
    pred_mask = cv2.resize(pred_mask, (2100, 1400))
    
    # Convert to binary masks (threshold at 0.5)
    pred_mask = (pred_mask > 0.5).astype(np.uint8)
    
    # Convert each channel to RLE and create submission rows
    for i, cloud_type in enumerate(CLOUD_TYPES):
        rle = mask_to_rle(pred_mask[:, :, i])
        submission_rows.append({
            'Image_Label': f"{img_name}_{cloud_type}",
            'EncodedPixels': rle if rle else '-1'
        })

print("Predictions complete!")

# Create Submission File

In [ ]:
# Create submission dataframe
submission_df = pd.DataFrame(submission_rows)

# Save to CSV
submission_df.to_csv('submission_fpn_resnet50.csv', index=False)

print(f"Submission file created: submission_fpn_resnet50.csv")
print(f"Shape: {submission_df.shape}")
print(f"\nFirst few rows:")
print(submission_df.head(10))

# Summary

## Model Architecture:
- **Backbone:** ResNet50 (pretrained on ImageNet)
- **Head:** Feature Pyramid Network (FPN)
- **Parameters:** ~35M (smaller than U-Net variants)
- **Input size:** 320×480×3 (resized from 1400×2100)
- **Output:** 320×480×4 (4 cloud types)

## Training:
- **Loss:** Binary Cross-Entropy + Dice Loss
- **Optimizer:** Adam (lr=1e-4)
- **Augmentation:** Flips, rotations, brightness, noise
- **Callbacks:** Early stopping, LR reduction, model checkpointing

## Results:
- **Validation Dice:** ~0.65-0.75 (expected)
- **Validation IoU:** ~0.55-0.65
- **Training time:** ~2-3 hours on GPU

## Advantages of FPN + ResNet:
1. **Multi-scale features:** Detects clouds of all sizes
2. **Transfer learning:** ResNet pretrained weights speed up convergence
3. **Efficient:** Faster than U-Net while maintaining accuracy
4. **Flexible:** Easy to swap backbone (ResNet50 → ResNet101, EfficientNet, etc.)

## Potential Improvements:
1. **Larger input size:** Use 512×768 or original 1400×2100
2. **Test-time augmentation (TTA):** Average predictions from augmented versions
3. **Deeper backbone:** ResNet101, ResNet152, or EfficientNet
4. **Post-processing:** Connected components, morphological operations
5. **Ensemble:** Combine FPN + U-Net predictions
6. **Class weights:** Handle class imbalance
7. **Focal loss:** Focus on hard examples

## Real-world analogy:
FPN is like having multiple detectives looking at different zoom levels:
- **P2 (high resolution):** Spots small clouds and fine details
- **P3-P4 (medium):** Identifies cloud shapes and patterns
- **P5 (low resolution):** Understands overall cloud distribution
All detectives share information (lateral connections) to make the final decision!